# Setup Tests

In [1]:
!nvidia-smi

Thu Oct  8 15:23:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. Use Runtime → Change runtime type → GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB",
)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA L4
GPU memory: 22.0 GB


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

archive = Path(
    "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab.tar"
)

print("Archive exists:", archive.exists())
print("Archive size GB:", round(archive.stat().st_size / 1024**3, 2))

assert archive.exists(), f"Archive not found: {archive}"

Archive exists: True
Archive size GB: 2.06


In [5]:
!mkdir -p /content/drone-detection-v3
!tar -xf "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab.tar" \
  -C /content/drone-detection-v3

In [6]:
%cd /content/drone-detection-v3

/content/drone-detection-v3


In [7]:
from pathlib import Path

required = [
    Path("tools/train_ubsod.py"),
    Path("splits/ub-sod-scene-v1/train.txt"),
    Path("splits/ub-sod-scene-v1/val.txt"),
    Path("splits/ub-sod-scene-v1/test.txt"),
    Path("data/UB-SOD-release/verified-tar/images"),
    Path("data/UB-SOD-release/verified-tar/labels"),
]

for path in required:
    print(path, "OK" if path.exists() else "MISSING")

assert all(path.exists() for path in required)

tools/train_ubsod.py OK
splits/ub-sod-scene-v1/train.txt OK
splits/ub-sod-scene-v1/val.txt OK
splits/ub-sod-scene-v1/test.txt OK
data/UB-SOD-release/verified-tar/images OK
data/UB-SOD-release/verified-tar/labels OK


In [8]:
!pip install -q "ultralytics==8.4.174"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 11.5 MB/s eta 0:00:00


In [9]:
import sys
import torch
import ultralytics

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

assert ultralytics.__version__ == "8.4.174"
assert torch.cuda.is_available()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.174
CUDA: True
GPU: NVIDIA L4


In [10]:
!python tools/train_ubsod.py --preflight-only

{
  "checked_at_utc": "2026-10-08T15:35:31.617569+00:00",
  "dataset_root": "/content/drone-detection-v3/data/UB-SOD-release/verified-tar",
  "split_root": "/content/drone-detection-v3/splits/ub-sod-scene-v1",
  "classes": {
    "0": "UAV",
    "1": "Bird"
  },
  "counts": {
    "train": 5367,
    "val": 876,
    "test": 875
  },
  "split_sha256": {
    "train": "78a1c9fe94229617b61fcd2a6082c41f639d2974eaeb9fa086c29a476688954d",
    "val": "bd7ef42705b146e2592e3b8fc7cc906e25e5446fd3c62a36ffa488472664a6de",
    "test": "74e3b8763d3da4578b001e26e7b4d85f5095636056f8ea3c46580dda955e0a07"
  },
  "missing_images_or_labels": 0,
  "invalid_labels": 0,
  "cross_split_filename_overlap": 0,
  "model": "yolo26n.pt",
  "imgsz": 640,
  "epochs": 100,
  "batch": -1,
  "fraction": 1.0,
  "save_period": 5,
  "device": null,
  "workers": 4,
  "seed": 42,
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "dataset_yaml": "/content/dro

In [11]:
!mkdir -p "/content/drive/MyDrive/drone-detection-v3/runs/environment"

!python -m pip freeze \
  > "/content/drive/MyDrive/drone-detection-v3/runs/environment/pip-freeze.txt"

!nvidia-smi \
  > "/content/drive/MyDrive/drone-detection-v3/runs/environment/nvidia-smi.txt"

In [13]:
import yaml
from pathlib import Path

# Path to the generated dataset config
yaml_path = Path("/content/drive/MyDrive/drone-detection-v3/runs/ubsod-scene-v1-yolo26n-smoke/dataset.generated.yaml")

if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    print("--- Generated Dataset YAML Configuration ---")
    print(yaml.dump(config))

    # Let's inspect paths listed in the split files
    for split in ['train', 'val']:
        if split in config:
            split_file = Path(config[split])
            print(f"\nFirst 3 entries in {split} split file ({split_file}):")
            if split_file.exists():
                with open(split_file, 'r') as sf:
                    lines = [sf.readline().strip() for _ in range(3)]
                    for line in lines:
                        print(f"  - {line}")
            else:
                print(f"  [ERROR] Split file does not exist!")
else:
    print(f"Could not find generated dataset YAML at {yaml_path}")

--- Generated Dataset YAML Configuration ---
names:
  0: UAV
  1: Bird
path: /content/drone-detection-v3/data/UB-SOD-release/verified-tar
train: /content/drone-detection-v3/splits/ub-sod-scene-v1/train.txt
val: /content/drone-detection-v3/splits/ub-sod-scene-v1/val.txt


First 3 entries in train split file (/content/drone-detection-v3/splits/ub-sod-scene-v1/train.txt):
  - images/000001.jpg
  - images/000002.jpg
  - images/000003.jpg

First 3 entries in val split file (/content/drone-detection-v3/splits/ub-sod-scene-v1/val.txt):
  - images/000000.jpg
  - images/000013.jpg
  - images/000015.jpg


In [14]:
# Verify the exact directory structures and counts for verification
images_dir = Path("/content/drone-detection-v3/data/UB-SOD-release/verified-tar/images")
labels_dir = Path("/content/drone-detection-v3/data/UB-SOD-release/verified-tar/labels")

print(f"Images directory exists: {images_dir.exists()}")
print(f"Labels directory exists: {labels_dir.exists()}")

if images_dir.exists():
    image_files = list(images_dir.glob("**/*"))
    image_files = [f for f in image_files if f.is_file()]
    print(f"Total images found: {len(image_files)}")
    if image_files:
        print(f"Sample image path: {image_files[0]}")

if labels_dir.exists():
    label_files = list(labels_dir.glob("**/*"))
    label_files = [f for f in label_files if f.is_file()]
    print(f"Total labels found: {len(label_files)}")
    if label_files:
        print(f"Sample label path: {label_files[0]}")

Images directory exists: True
Labels directory exists: True
Total images found: 7118
Sample image path: /content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/003096.jpg
Total labels found: 7118
Sample label path: /content/drone-detection-v3/data/UB-SOD-release/verified-tar/labels/000194.txt


In [15]:
from pathlib import Path

# We will update the split files to use absolute paths
dataset_root = Path("/content/drone-detection-v3/data/UB-SOD-release/verified-tar")
split_dir = Path("/content/drone-detection-v3/splits/ub-sod-scene-v1")

for split_name in ["train", "val", "test"]:
    split_file = split_dir / f"{split_name}.txt"
    if split_file.exists():
        with open(split_file, "r") as f:
            lines = f.read().splitlines()

        # Convert each relative line (e.g., 'images/000001.jpg') to an absolute path
        absolute_lines = []
        for line in lines:
            abs_path = (dataset_root / line).resolve()
            absolute_lines.append(str(abs_path))

        with open(split_file, "w") as f:
            f.write("\n".join(absolute_lines) + "\n")

        print(f"Updated {split_name}.txt with absolute paths. First entry:")
        print(f"  {absolute_lines[0]}")

Updated train.txt with absolute paths. First entry:
  /content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000001.jpg
Updated val.txt with absolute paths. First entry:
  /content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000000.jpg
Updated test.txt with absolute paths. First entry:
  /content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000006.jpg


In [16]:
%%bash
set -o pipefail

RUN_ROOT="/content/drive/MyDrive/drone-detection-v3/runs"
RUN_NAME="ubsod-scene-v1-yolo26n-smoke"

mkdir -p "${RUN_ROOT}/${RUN_NAME}"

# Run training with the updated absolute paths in the split files
python tools/train_ubsod.py \
  --device 0 \
  --epochs 1 \
  --fraction 0.1 \
  --batch -1 \
  --workers 2 \
  --imgsz 640 \
  --project "${RUN_ROOT}" \
  --name "${RUN_NAME}" \
  2>&1 | tee "${RUN_ROOT}/${RUN_NAME}/console.log"

{
  "checked_at_utc": "2026-10-08T15:41:00.276477+00:00",
  "dataset_root": "/content/drone-detection-v3/data/UB-SOD-release/verified-tar",
  "split_root": "/content/drone-detection-v3/splits/ub-sod-scene-v1",
  "classes": {
    "0": "UAV",
    "1": "Bird"
  },
  "counts": {
    "train": 5367,
    "val": 876,
    "test": 875
  },
  "split_sha256": {
    "train": "4759814d333a9edafd99e300cd3a5fed84f521b67763103f2e88c3cfc128102f",
    "val": "f919009e53a7dcd0e03872510daeebba5a2898431837dd65abdfab1c2f09a4dc",
    "test": "ca04304e395061c13a7392229a24d702326e29815b6d92abb63e9be564fa7658"
  },
  "missing_images_or_labels": 0,
  "invalid_labels": 0,
  "cross_split_filename_overlap": 0,
  "model": "yolo26n.pt",
  "imgsz": 640,
  "epochs": 1,
  "batch": -1,
  "fraction": 0.1,
  "save_period": 5,
  "device": "0",
  "workers": 2,
  "seed": 42,
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "dataset_yaml": "/content/drive/

In [17]:
from pathlib import Path
import hashlib

split_root = Path(
    "/content/drone-detection-v3/splits/ub-sod-scene-v1"
)

for split in ("train", "val", "test"):
    path = split_root / f"{split}.txt"

    entries = [
        line.strip().replace("\\", "/")
        for line in path.read_text().splitlines()
        if line.strip()
    ]

    canonical_content = "\n".join(sorted(entries)) + "\n"
    canonical_hash = hashlib.sha256(
        canonical_content.encode("utf-8")
    ).hexdigest()

    print(split, len(entries), canonical_hash)

train 5367 4759814d333a9edafd99e300cd3a5fed84f521b67763103f2e88c3cfc128102f
val 876 f919009e53a7dcd0e03872510daeebba5a2898431837dd65abdfab1c2f09a4dc
test 875 ca04304e395061c13a7392229a24d702326e29815b6d92abb63e9be564fa7658


In [18]:
from pathlib import Path

run = Path(
    "/content/drive/MyDrive/drone-detection-v3/runs/"
    "ubsod-scene-v1-yolo26n-smoke"
)

for relative in (
    "weights/best.pt",
    "weights/last.pt",
    "results.csv",
    "args.yaml",
    "preflight.json",
    "console.log",
    "results.png",
):
    path = run / relative
    print(relative, "OK" if path.exists() else "MISSING")

weights/best.pt OK
weights/last.pt OK
results.csv OK
args.yaml OK
preflight.json OK
console.log OK
results.png OK


# Added a hard split-membership gate to *train_ubsod*.py

In [37]:
!sha256sum "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix.tar"

f82db7e18801015b32c08249ef397af46dfc36ac3622048b28fb6bf9ec84a6a2  /content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix.tar


In [44]:
!tar -xvf \
  "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix-v2.tar" \
  -C /content/drone-detection-v3

tools/train_ubsod.py
splits/ub-sod-scene-v1/
splits/ub-sod-scene-v1/README.md
splits/ub-sod-scene-v1/split_manifest.json
splits/ub-sod-scene-v1/test.txt
splits/ub-sod-scene-v1/train.txt
splits/ub-sod-scene-v1/val.txt


In [45]:
!grep -n "train.resolved.txt" /content/drone-detection-v3/tools/train_ubsod.py

122:        f"train: {json.dumps((run_dir / 'train.resolved.txt').as_posix())}",
159:    write_resolved_split(args.dataset_root, args.split_root / "train.txt", run_dir / "train.resolved.txt")


In [46]:
!ls -l /content/drone-detection-v3/tools/train_ubsod.py

-rw-rw-rw- 1 root root 9282 Oct  8 16:23 /content/drone-detection-v3/tools/train_ubsod.py


In [47]:
%cd /content/drone-detection-v3

/content/drone-detection-v3


In [48]:
!python tools/train_ubsod.py \
  --preflight-only \
  --project "/content/drive/MyDrive/drone-detection-v3/runs" \
  --name "ubsod-scene-v1-yolo26n-smoke-corrected-v2"

{
  "checked_at_utc": "2026-10-08T16:29:47.445621+00:00",
  "dataset_root": "/content/drone-detection-v3/data/UB-SOD-release/verified-tar",
  "split_root": "/content/drone-detection-v3/splits/ub-sod-scene-v1",
  "classes": {
    "0": "UAV",
    "1": "Bird"
  },
  "counts": {
    "train": 5367,
    "val": 876,
    "test": 875
  },
  "split_sha256": {
    "train": "78a1c9fe94229617b61fcd2a6082c41f639d2974eaeb9fa086c29a476688954d",
    "val": "bd7ef42705b146e2592e3b8fc7cc906e25e5446fd3c62a36ffa488472664a6de",
    "test": "74e3b8763d3da4578b001e26e7b4d85f5095636056f8ea3c46580dda955e0a07"
  },
  "canonical_split_sha256": {
    "train": "2e6100e2c3f9cd5019f45bd90b4090cc7bc5ee270ae97d020c21245c86832921",
    "val": "c6215299af7eedc0baf70e394b64ce18519f0937be267a433568a1008465927a",
    "test": "e931bf035b9e091b019ad3c0576fe36dd05e8d2be774939ce4e181c1a3445d92"
  },
  "missing_images_or_labels": 0,
  "invalid_labels": 0,
  "cross_split_filename_overlap": 0,
  "model": "yolo26n.pt",
  "imgsz": 6

In [42]:
!ls -lh "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix-v2.tar"
!sha256sum "/content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix-v2.tar"

-rw------- 1 root root 325K Oct  8 16:24 /content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix-v2.tar
77e0ead4829ebd9d3036b472a8af5d6d9bc24c7a62b180df954b7214cfecdcd0  /content/drive/MyDrive/drone-detection-v3/input/ubsod-colab-split-fix-v2.tar


In [49]:
!head -n 3 \
  "/content/drive/MyDrive/drone-detection-v3/runs/ubsod-scene-v1-yolo26n-smoke-corrected-v2/train.resolved.txt"

/content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000001.jpg
/content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000002.jpg
/content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000003.jpg


In [51]:
%%bash
FIRST_IMAGE=$(head -n 1 "/content/drive/MyDrive/drone-detection-v3/runs/ubsod-scene-v1-yolo26n-smoke-corrected-v2/train.resolved.txt")
echo "$FIRST_IMAGE"
test -f "$FIRST_IMAGE" && echo "IMAGE EXISTS" || echo "IMAGE MISSING"

/content/drone-detection-v3/data/UB-SOD-release/verified-tar/images/000001.jpg
IMAGE EXISTS


In [52]:
%%bash
set -o pipefail

cd /content/drone-detection-v3

RUN_ROOT="/content/drive/MyDrive/drone-detection-v3/runs"
RUN_NAME="ubsod-scene-v1-yolo26n-smoke-corrected-v2"

python tools/train_ubsod.py \
  --device 0 \
  --epochs 1 \
  --fraction 0.1 \
  --batch -1 \
  --workers 2 \
  --imgsz 640 \
  --project "${RUN_ROOT}" \
  --name "${RUN_NAME}" \
  2>&1 | tee "${RUN_ROOT}/${RUN_NAME}/console.log"

{
  "checked_at_utc": "2026-10-08T16:31:04.068305+00:00",
  "dataset_root": "/content/drone-detection-v3/data/UB-SOD-release/verified-tar",
  "split_root": "/content/drone-detection-v3/splits/ub-sod-scene-v1",
  "classes": {
    "0": "UAV",
    "1": "Bird"
  },
  "counts": {
    "train": 5367,
    "val": 876,
    "test": 875
  },
  "split_sha256": {
    "train": "78a1c9fe94229617b61fcd2a6082c41f639d2974eaeb9fa086c29a476688954d",
    "val": "bd7ef42705b146e2592e3b8fc7cc906e25e5446fd3c62a36ffa488472664a6de",
    "test": "74e3b8763d3da4578b001e26e7b4d85f5095636056f8ea3c46580dda955e0a07"
  },
  "canonical_split_sha256": {
    "train": "2e6100e2c3f9cd5019f45bd90b4090cc7bc5ee270ae97d020c21245c86832921",
    "val": "c6215299af7eedc0baf70e394b64ce18519f0937be267a433568a1008465927a",
    "test": "e931bf035b9e091b019ad3c0576fe36dd05e8d2be774939ce4e181c1a3445d92"
  },
  "missing_images_or_labels": 0,
  "invalid_labels": 0,
  "cross_split_filename_overlap": 0,
  "model": "yolo26n.pt",
  "imgsz": 6

# full YOLO26n run

In [53]:
%%bash
set -o pipefail

cd /content/drone-detection-v3

RUN_ROOT="/content/drive/MyDrive/drone-detection-v3/runs"
RUN_NAME="ubsod-scene-v1-yolo26n-640-seed42"

mkdir -p "${RUN_ROOT}/${RUN_NAME}"

python tools/train_ubsod.py \
  --device 0 \
  --epochs 100 \
  --fraction 1.0 \
  --batch -1 \
  --workers 2 \
  --imgsz 640 \
  --save-period 5 \
  --seed 42 \
  --project "${RUN_ROOT}" \
  --name "${RUN_NAME}" \
  2>&1 | tee "${RUN_ROOT}/${RUN_NAME}/console.log"

{
  "checked_at_utc": "2026-10-08T16:36:51.359330+00:00",
  "dataset_root": "/content/drone-detection-v3/data/UB-SOD-release/verified-tar",
  "split_root": "/content/drone-detection-v3/splits/ub-sod-scene-v1",
  "classes": {
    "0": "UAV",
    "1": "Bird"
  },
  "counts": {
    "train": 5367,
    "val": 876,
    "test": 875
  },
  "split_sha256": {
    "train": "78a1c9fe94229617b61fcd2a6082c41f639d2974eaeb9fa086c29a476688954d",
    "val": "bd7ef42705b146e2592e3b8fc7cc906e25e5446fd3c62a36ffa488472664a6de",
    "test": "74e3b8763d3da4578b001e26e7b4d85f5095636056f8ea3c46580dda955e0a07"
  },
  "canonical_split_sha256": {
    "train": "2e6100e2c3f9cd5019f45bd90b4090cc7bc5ee270ae97d020c21245c86832921",
    "val": "c6215299af7eedc0baf70e394b64ce18519f0937be267a433568a1008465927a",
    "test": "e931bf035b9e091b019ad3c0576fe36dd05e8d2be774939ce4e181c1a3445d92"
  },
  "missing_images_or_labels": 0,
  "invalid_labels": 0,
  "cross_split_filename_overlap": 0,
  "model": "yolo26n.pt",
  "imgsz": 6